In [1]:
!pip install fastapi uvicorn pydantic
!pip install mlflow scikit-learn fastapi uvicorn pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [2]:
!umount -l /content/drive 2>/dev/null
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv('/content/drive/MyDrive/f1_mlops/data/processed/f1_dataset_clean.csv')
df = df.dropna(subset=['positionOrder'])
df['is_points'] = (df['positionOrder'] <= 10).astype(int)

X_train, y_train = df[df['year'] < 2022][['grid', 'year']], df[df['year'] < 2022]['is_points']
X_test, y_test = df[df['year'] == 2022][['grid', 'year']], df[df['year'] == 2022]['is_points']

mlflow.set_tracking_uri("file:///content/drive/MyDrive/f1_mlops/mlruns")
mlflow.set_experiment("F1_Model_Competition")

diccionario_modelos = {
    "Regresion_Logistica": {
        "estimador": LogisticRegression(max_iter=1000),
        "grid_params": {"C": [0.1, 1.0, 10.0]}
    },
    "Random_Forest": {
        "estimador": RandomForestClassifier(random_state=42),
        "grid_params": {"n_estimators": [50, 100], "max_depth": [5, 10]}
    },
    "Gradient_Boosting": {
        "estimador": GradientBoostingClassifier(random_state=42),
        "grid_params": {"n_estimators": [50, 100], "learning_rate": [0.01, 0.1]}
    }
}

for nombre_modelo, config in diccionario_modelos.items():
    print(f"Entrenando y optimizando: {nombre_modelo}...")

    with mlflow.start_run(run_name=nombre_modelo):
        buscador = GridSearchCV(config["estimador"], config["grid_params"], cv=3, scoring='accuracy')
        buscador.fit(X_train, y_train)

        mejor_modelo = buscador.best_estimator_

        y_pred = mejor_modelo.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)

        mlflow.log_params(buscador.best_params_)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.sklearn.log_model(mejor_modelo, "modelo_f1")

        print(f"{nombre_modelo} finalizado -> Accuracy: {acc:.4f} | Mejores Params: {buscador.best_params_}\n")

runs = mlflow.search_runs(experiment_names=["F1_Model_Competition"])
ganador = runs.sort_values("metrics.accuracy", ascending=False).iloc[0]

print("Mejor modelo:")
print(f"Modelo: {ganador['tags.mlflow.runName']}")
print(f"Accuracy: {ganador['metrics.accuracy']:.4f}")
print(f"ID del Run: {ganador['run_id']}")

Entrenando y optimizando: Regresion_Logistica...


2026/05/01 10:13:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 10:13:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Regresion_Logistica finalizado -> Accuracy: 0.7182 | Mejores Params: {'C': 0.1}

Entrenando y optimizando: Random_Forest...


2026/05/01 10:13:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 10:13:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random_Forest finalizado -> Accuracy: 0.7318 | Mejores Params: {'max_depth': 5, 'n_estimators': 50}

Entrenando y optimizando: Gradient_Boosting...


2026/05/01 10:13:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 10:13:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Gradient_Boosting finalizado -> Accuracy: 0.7318 | Mejores Params: {'learning_rate': 0.01, 'n_estimators': 100}

Mejor modelo:
Modelo: Gradient_Boosting
Accuracy: 0.7318
ID del Run: 2834e57b081b40f4be9322aa8dfba03b


In [4]:
import os
import sys
import importlib
import json
from fastapi.testclient import TestClient

ruta_base = '/content/drive/MyDrive/f1_mlops/mlruns'
ruta_modelo = None

for root, dirs, files in os.walk(ruta_base):
    if 'MLmodel' in files and 'model.pkl' in files:
        ruta_modelo = root
        break

if ruta_modelo is None:
    print("ERROR: No se ha encontrado el modelo. Parece que en los reinicios anteriores no se guardó. Vuelve a ejecutar la celda de Entrenamiento (Fase 2).")
else:
    print(f"Modelo encontrado -> Ruta exacta: {ruta_modelo}\n")

    codigo_api = f"""from fastapi import FastAPI
from pydantic import BaseModel
import mlflow
import pandas as pd

class PilotoInput(BaseModel):
    grid: int
    year: int

app = FastAPI(title="API Predictiva F1 - MLOps")
modelo_f1 = None

@app.on_event("startup")
def cargar_modelo():
    global modelo_f1
    try:
        ruta_directa = "file://{ruta_modelo}"
        modelo_f1 = mlflow.sklearn.load_model(ruta_directa)
        print("Modelo cargado desde el disco duro")
    except Exception as e:
        print(f"Error al cargar el modelo: {{e}}")

@app.post("/predecir")
def hacer_prediccion(datos: PilotoInput):
    if modelo_f1 is None:
        return {{"error": "El modelo no está cargado"}}

    df_entrada = pd.DataFrame([{{"grid": datos.grid, "year": datos.year}}])
    prediccion = modelo_f1.predict(df_entrada)[0]
    resultado_texto = "Sí (Top 10)" if prediccion == 1 else "No (Fuera de los puntos)"

    return {{
        "posicion_salida": datos.grid,
        "año_carrera": datos.year,
        "prediccion_modelo": resultado_texto
    }}
"""

    with open('/content/drive/MyDrive/f1_mlops/src/api.py', 'w') as f:
        f.write(codigo_api)

    sys.path.append('/content/drive/MyDrive/f1_mlops')
    if 'src.api' in sys.modules:
        importlib.reload(sys.modules['src.api'])

    from src.api import app

    with TestClient(app) as client:
        datos_json = {"grid": 3, "year": 2024}
        respuesta = client.post("/predecir", json=datos_json)

        if respuesta.status_code == 200 and "error" not in respuesta.json():
            print("-" * 40)
            print("RESPUESTA DE LA API:")
            print(json.dumps(respuesta.json(), indent=2, ensure_ascii=False))
            print("\n La API funciona perfectamente.")
        else:
            print("Error en la respuesta:", respuesta.text)

Modelo encontrado -> Ruta exacta: /content/drive/MyDrive/f1_mlops/mlruns/505708190338931035/models/m-d99fa34fd5684fe294dca9e97467a395/artifacts

Modelo cargado desde el disco duro
----------------------------------------
RESPUESTA DE LA API:
{
  "posicion_salida": 3,
  "año_carrera": 2024,
  "prediccion_modelo": "Sí (Top 10)"
}

 La API funciona perfectamente.


In [11]:
from google.colab import output
import time

print("🧹 Limpiando...")
!pkill -f uvicorn
time.sleep(2)

print("🚀 Encendiendo la API de FastAPI...")
# Ejecutamos Uvicorn en el puerto 8000
get_ipython().system_raw('nohup uvicorn src.api:app --app-dir /content/drive/MyDrive/f1_mlops --host 0.0.0.0 --port 8000 > api.log 2>&1 &')
time.sleep(5) # Damos tiempo a que cargue el modelo de MLflow

print("✅ ¡API LISTA! Haz clic en el enlace de abajo para abrirla en tu navegador:")
# Esta es la magia de Google Colab para abrir puertos
output.serve_kernel_port_as_window(8000)

🧹 Limpiando...
🚀 Encendiendo la API de FastAPI...
✅ ¡API LISTA! Haz clic en el enlace de abajo para abrirla en tu navegador:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>